# Cost & Token Observability

One LLM call is cheap. Ten thousand calls a day across five features is a
budget line you have to defend. This notebook shows how the Gateway turns
every call into one queryable record — tokens, cost, latency, feature — so
"what does this cost?" and "why is it slow?" have answers.

## The single source of truth: PRICING

Per-model prices live in one place (`pricing.py`). `calc_cost_usd` is the only
function that turns token counts into dollars, so there is no chance of two
parts of the system disagreeing on price.

In [1]:
import sys
sys.path.insert(0, "../src")

from genai_prod_kit.pricing import PRICING, calc_cost_usd

for model, p in PRICING.items():
    print(f"{model:24s}  in ${p['input_per_1m_usd']}/1M  out ${p['output_per_1m_usd']}/1M")

gemini-2.5-flash          in $0.3/1M  out $2.5/1M
gemini-2.5-pro            in $1.25/1M  out $10.0/1M
gemini-embedding-001      in $0.15/1M  out $0.0/1M


## Cost is computed, not guessed

Given token counts, the cost is deterministic. An unknown model returns 0.0
rather than crashing — pricing must never break a live call.

In [2]:
print("flash, 1M+1M tokens :", calc_cost_usd("gemini-2.5-flash", 1_000_000, 1_000_000), "USD")
print("pro,   1M+1M tokens :", calc_cost_usd("gemini-2.5-pro",   1_000_000, 1_000_000), "USD")
print("unknown model       :", calc_cost_usd("does-not-exist",   1_000_000, 1_000_000), "USD")

flash, 1M+1M tokens : 2.8 USD
pro,   1M+1M tokens : 11.25 USD
unknown model       : 0.0 USD


## Every call writes one record

The Gateway wraps the provider call, measures latency, computes cost, and
emits one `InvocationRecord` to a sink. Here the sink is local JSON Lines — no
cloud required.

In [3]:
import os
from genai_prod_kit.gateway import call_llm
from genai_prod_kit.providers.gemini import GeminiProvider
from genai_prod_kit.sinks.jsonl import JsonlSink

provider = GeminiProvider(os.environ["GEMINI_API_KEY"])
sink = JsonlSink("nb02_invocations.jsonl")

for q in ["What is 2+2?", "Name one ocean.", "Capital of France?"]:
    call_llm(q, provider=provider, sink=sink,
             feature="demo_batch", model="gemini-2.5-flash")
print("3 calls recorded to nb02_invocations.jsonl")

3 calls recorded to nb02_invocations.jsonl


## Reading the telemetry back

The records are plain JSON Lines — query them with anything. Here we total the
cost and average the latency over the batch we just ran.

In [4]:
import json

rows = [json.loads(l) for l in open("nb02_invocations.jsonl", encoding="utf-8") if l.strip()]
total_cost = sum(r["estimated_cost_usd"] for r in rows)
avg_latency = sum(r["latency_ms"] for r in rows) / len(rows)
total_out = sum(r["output_tokens"] for r in rows)

print(f"calls        : {len(rows)}")
print(f"total cost   : ${total_cost:.6f}")
print(f"avg latency  : {avg_latency:.0f} ms")
print(f"output tokens: {total_out}")

calls        : 3
total cost   : $0.000033
avg latency  : 1136 ms
output tokens: 11


## Why this matters

With one record per call you can answer, per feature: monthly spend, p95
latency, token growth over time, and error rate. None of that is possible when
the provider SDK is called directly and the result is thrown away. The Gateway
is the seam where observability becomes free.